# Japanese–English Direct S2ST corpus — Colab Pro

このノートブックはGoogle Driveへ成果物とcheckpointを永続保存し、Colabの切断後も未完了shardから再開します。最初に **ランタイム → ランタイムのタイプを変更 → GPU** を選択してください。L4/A100を推奨し、T4ではFP16へ自動的に切り替わります。

In [ ]:
# 1. Google Driveをマウントし、永続保存先を設定
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

REPO_URL = 'https://github.com/Yaaamashiro/ja-en-direct-s2st-corpus.git'
REPO_DIR = Path('/content/ja-en-direct-s2st-corpus')
DATA_ROOT = Path('/content/drive/MyDrive/ja-en-direct-s2st-corpus-data')
HF_HOME = DATA_ROOT / '.cache' / 'huggingface'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ['S2ST_DATA_ROOT'] = str(DATA_ROOT)
os.environ['HF_HOME'] = str(HF_HOME)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Data:', DATA_ROOT)
print('Model cache:', HF_HOME)

In [ ]:
# 2. 最新コードを取得
import subprocess
import sys
PYTHON = sys.executable

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
print('Revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
# 3. システム・Python依存関係を導入（ランタイム作成ごとに1回）
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(
    ['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'libsndfile1', 'sox'],
    check=True,
)
subprocess.run(
    [
        PYTHON, '-m', 'pip', 'install', '-q',
        'torch==2.7.1', 'torchaudio==2.7.1',
        '--index-url', 'https://download.pytorch.org/whl/cu126',
    ],
    check=True,
)
subprocess.run(
    [PYTHON, '-m', 'pip', 'install', '-q', '-r', 'requirements/runtime.txt'],
    check=True,
)
subprocess.run(
    [PYTHON, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'],
    check=True,
)
print('Dependencies installed.')

In [ ]:
# 4. GPUを確認。ここで失敗した場合はGPUランタイムへ変更
import torch

assert torch.cuda.is_available(), 'GPUが見つかりません。GPUランタイムへ変更してください。'
props = torch.cuda.get_device_properties(0)
vram_gib = props.total_memory / 1024**3
dtype = 'bfloat16' if torch.cuda.is_bf16_supported() else 'float16'
print({'gpu': props.name, 'vram_gib': round(vram_gib, 2), 'tts_dtype': dtype})
assert vram_gib >= 14, 'VRAMが不足しています。T4（約14.7 GiB）以上が必要です。'

## 初回だけ：5文スモークテスト

公式JESC/KFTTを準備した後、日英10音声の生成とWhisper検査を行います。切断された場合は同じセルをもう一度実行してください。

In [ ]:
CONFIG = REPO_DIR / 'configs' / 'colab-pro.yaml'
subprocess.run(
    [PYTHON, '-m', 's2st_corpus.cli', '--config', str(CONFIG), 'smoke-test'],
    check=True,
)

## 本番：未完了shardを1つ処理

スモークテストの音声とmanifestを確認してから実行してください。セルを実行するたびに、最初の未完了shardを1つ処理します。途中切断後も同じshard内のcheckpointから再開します。

In [ ]:
# 現在の進捗
subprocess.run(
    [PYTHON, '-m', 's2st_corpus.cli', '--config', str(CONFIG), 'status'],
    check=True,
)

In [ ]:
# 次の未完了shardを実行。完了後、必要に応じてこのセルを再実行
subprocess.run(
    [PYTHON, '-m', 's2st_corpus.cli', '--config', str(CONFIG), 'run-next-shard'],
    check=True,
)

## 集計

途中経過は `--allow-incomplete` で集計できます。全512 shardの完了後は最後のセルで正式なrelease manifestを作成します。

In [ ]:
# 任意：現在までの途中集計
subprocess.run(
    [
        PYTHON, '-m', 's2st_corpus.cli', '--config', str(CONFIG),
        'consolidate', '--allow-incomplete',
    ],
    check=True,
)

In [ ]:
# 全shard完了後のみ：正式な最終集計
subprocess.run(
    [PYTHON, '-m', 's2st_corpus.cli', '--config', str(CONFIG), 'consolidate'],
    check=True,
)